In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wfwgriw_/unsloth_7b90afd38d3e414499c781ae6d1f29e8
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wfwgriw_/unsloth_7b90afd38d3e414499c781ae6d1f29e8
  Resolved https://github.com/unslothai/unsloth.git to commit efed5c37394a144349cd9b1ea525e132e04584e5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

DRIVE_BASE = "/content/drive/MyDrive/ChefGPT"
LOCAL_QA = f"{DRIVE_BASE}/chefgpt_qa_clean-50k.jsonl"
DRIVE_CHECKPOINTS = f"{DRIVE_BASE}/checkpoints"
DRIVE_GGUF = f"{DRIVE_BASE}/chefgpt-14b-gguf"
LOCAL_CHECKPOINTS = "/content/checkpoints" # <--- Eksik olan tanım buraya eklendi

os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)
print("Veri yolu hazır!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Veri yolu hazır!


In [4]:
from unsloth import FastLanguageModel
import torch

# 14B Milyar Parametreli Modele Geçildi
MODEL_NAME = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    dtype=None, load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"✅ Model hazir: {MODEL_NAME}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

unsloth/Qwen2.5-14B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 48 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ Model hazir: unsloth/Qwen2.5-14B-Instruct-bnb-4bit


In [5]:
from datasets import load_dataset

# Sorun çözüldü, standart, güvenli ve hızlı load_dataset yöntemine geri dönüldü.
dataset = load_dataset("json", data_files=LOCAL_QA, split="train")

def format_chat(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_chat, batched=False).shuffle(seed=42)
print(f"✅ Hazir: {len(dataset)} ornek (İçinde sentetik sohbet verileri mevcut!)")



✅ Hazir: 51500 ornek (İçinde sentetik sohbet verileri mevcut!)


In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback
import os, shutil

class DriveCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        src = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        dst = os.path.join(DRIVE_CHECKPOINTS, f"checkpoint-{state.global_step}")
        if os.path.exists(src):
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(src, dst)
            all_c = sorted([d for d in os.listdir(DRIVE_CHECKPOINTS) if d.startswith("checkpoint-")],
                          key=lambda x: int(x.split("-")[1]))
            while len(all_c) > 3:
                shutil.rmtree(os.path.join(DRIVE_CHECKPOINTS, all_c.pop(0)))

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=8, packing=True, # Veri işleme çekirdeği 8'e çıkarıldı
    args=TrainingArguments(
        output_dir=LOCAL_CHECKPOINTS,
        # NİTRO AYARLARI (Hız odaklı)
        per_device_train_batch_size=4,     # A100 VRAM'ine uygun maksimum güvenli batch
        gradient_accumulation_steps=4,    # Efektif batch size = 16
        num_train_epochs=1,
        learning_rate=5e-5,
        warmup_steps=50,                  # Isınma adımı azaltıldı
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=50,                  # HIZ İÇİN: Ekrana yazma sıklığı 50'ye çıkarıldı
        save_steps=500,                    # HIZ İÇİN: Google Drive'a kaydetme sıklığı 500'e çıkarıldı (Disk I/O yavaşlatıyordu)
        save_total_limit=2,
        optim="adamw_8bit", weight_decay=0.01, seed=42, report_to="none",
    ),
    callbacks=[DriveCallback()],
)

print(f"🚀 14B EĞİTİMİ BAŞLIYOR (A100 NİTRO MOD)...")
stats = trainer.train()



Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/51500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 14B EĞİTİMİ BAŞLIYOR (A100 NİTRO MOD)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,500 | Num Epochs = 1 | Total steps = 3,219
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
50,2.141381
100,0.859974
150,0.697175
200,0.669932
250,0.619186
300,0.591408


Step,Training Loss
50,2.141381
100,0.859974
150,0.697175
200,0.669932
250,0.619186
300,0.591408
350,0.593889
400,0.567800
450,0.541879
500,0.529927


Unsloth: Restored added_tokens_decoder metadata in /content/checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/checkpoints/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/checkpoints/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/checkpoints/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/checkpoints/checkpoint-2500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/checkpoints/checkpoint-3000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/checkpoints/checkpoint-3219/tokenizer_config.json.


In [8]:
LOCAL_GGUF = "/content/chefgpt-14b-gguf"
DRIVE_GGUF = "/content/drive/MyDrive/ChefGPT/chefgpt-14b-gguf"

model.save_pretrained_gguf(
    LOCAL_GGUF,
    tokenizer,
    quantization_method="q4_k_m"
)

import shutil, os

if os.path.exists(DRIVE_GGUF):
    shutil.rmtree(DRIVE_GGUF)

shutil.copytree(LOCAL_GGUF, DRIVE_GGUF)

print("✅ GGUF model localde üretildi ve Drive'a kopyalandı.")

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /content/chefgpt-14b-gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00006.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00006-of-00006.safetensors:   0%|          | 0.00/4.73G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 6/6 [01:15<00:00, 12.66s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 6/6 [02:00<00:00, 20.02s/it]


Unsloth: Merge process complete. Saved to `/content/chefgpt-14b-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/chefgpt-14b-gguf_gguf/Qwen2.5-14B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/chefgpt-14b-gguf_gguf/Qwen2.5-14B-Instruct.Q4_K_M.gguf']
Unsloth: 